In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/train.csv')
df['file_name'] = df['file_name'].str.replace('train_data', 'data/train_data')

In [4]:
# Long execution, can load from csv

def extract_image_features(file_path):
    bgr = cv2.imread(file_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Convert to HLS, HSV, LAB, and YCrCb
    hls = cv2.cvtColor(rgb, cv2.COLOR_RGB2HLS_FULL)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV_FULL)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    ycrcb = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)

    h, w, _ = rgb.shape
    mean_rgb = np.mean(rgb, axis=(0, 1))
    mean_hls = np.mean(hls, axis=(0, 1))
    mean_v = np.mean(hsv[:, :, 2])
    mean_lab = np.mean(lab, axis=(0, 1))
    mean_ycrcb = np.mean(ycrcb, axis=(0, 1))
    return [h, w, *mean_rgb, *mean_hls, mean_v, *mean_lab, *mean_ycrcb]

columns = ['height', 'width', 'mean_R', 'mean_G', 'mean_B', 'mean_H', 'mean_L', 'mean_S', 'mean_V', 'mean_LAB_L', 'mean_LAB_A', 'mean_LAB_B', 'mean_Y', 'mean_Cr', 'mean_Cb']
df[columns] = df['file_name'].apply(lambda path: pd.Series(extract_image_features(path)))

df_human = df[df['label']==0]
df_ai = df[df['label']==1]
#display(df_human) # 0 = human, 1 = AI
#display(df_ai) # 0 = human, 1 = AI

display(df_human.describe())
display(df_ai.describe())

def get_normalized_channel_histograms(file_path):
    bgr = cv2.imread(file_path)
    if bgr is None:
        raise ValueError(f"Image could not be read: {file_path}")

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    hls = cv2.cvtColor(rgb, cv2.COLOR_RGB2HLS)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    r, g, b = cv2.split(rgb)
    h, l, s = cv2.split(hls)
    v = hsv[:, :, 2]

    def norm_hist(channel):
        hist, _ = np.histogram(channel, bins=256, range=(0, 256))
        return hist / hist.sum()

    hist_df = pd.DataFrame({
        'R': norm_hist(r),
        'G': norm_hist(g),
        'B': norm_hist(b),
        'H': norm_hist(h),
        'L': norm_hist(l),
        'S': norm_hist(s),
        'V': norm_hist(v)
    }, index=np.arange(256))

    return hist_df

def compute_histograms_by_label(df):
    histograms = {0: [], 1: []}

    for _, row in df.iterrows():
        file_path = row['file_name']
        label = row['label']
        hist_df = get_normalized_channel_histograms(file_path)
        histograms[label].append(hist_df)


    avg_hist_0 = pd.concat(histograms[0]).groupby(level=0).mean() if histograms[0] else None
    avg_hist_1 = pd.concat(histograms[1]).groupby(level=0).mean() if histograms[1] else None

    return avg_hist_0, avg_hist_1

def plot_each_channel_by_label_separately(avg_hist_0, avg_hist_1):
    channels = avg_hist_0.columns
    fig, axs = plt.subplots(nrows=7, ncols=2, figsize=(14, 20), sharex=True, sharey=True)

    for i, channel in enumerate(channels):
        axs[i, 0].plot(avg_hist_0.index, avg_hist_0[channel], color='tab:blue')
        axs[i, 0].set_title(f'{channel} - Label 0')
        axs[i, 0].set_ylabel("Proportion of Pixels")
        axs[i, 0].grid(True)

        axs[i, 1].plot(avg_hist_1.index, avg_hist_1[channel], color='tab:orange')
        axs[i, 1].set_title(f'{channel} - Label 1')
        axs[i, 1].grid(True)

    for ax in axs[-1]:
        ax.set_xlabel("Pixel Value (0–255)")

    plt.tight_layout()
    plt.suptitle("Channel-wise Histograms by Label", fontsize=18, y=1.02)
    plt.show()

avg_hist_0, avg_hist_1 = compute_histograms_by_label(df)
avg_hist_0.to_csv("avg_hist_0.csv", index=True)
avg_hist_1.to_csv("avg_hist_1.csv", index=True)

,Unnamed: 0,label,height,width,mean_R,mean_G,mean_B,mean_H,mean_L,mean_S,mean_V,mean_LAB_L,mean_LAB_A,mean_LAB_B,mean_Y,mean_Cr,mean_Cb
count,39975.000000,39975.0,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000
mean,39975.000000,0.0,569.553171,716.566654,156.803924,147.628319,136.061878,76.685995,146.776671,89.925531,169.534170,155.812541,131.275920,135.933158,149.056143,133.533449,120.663378
std,23079.865684,0.0,118.956644,101.020640,49.825308,46.040022,51.716847,43.236724,46.116225,47.567583,45.391519,45.009062,8.470430,13.448598,45.685650,13.343410,13.941145
min,1.000000,0.0,112.000000,320.000000,1.417531,0.559710,0.786140,0.000000,2.360855,0.000000,3.063568,1.436518,66.284365,55.136851,1.742449,31.092480,13.891088
25%,19988.000000,0.0,512.000000,768.000000,122.062265,116.780938,98.112755,41.795277,114.490928,54.521532,139.765800,126.219093,127.598007,128.394836,117.610926,127.586357,114.172743
50%,39975.000000,0.0,512.000000,768.000000,158.676729,148.468516,137.466743,71.555748,146.136292,82.787417,172.986697,157.293921,130.327924,134.334330,149.183668,132.436019,122.171806
75%,59962.000000,0.0,640.000000,768.000000,196.142964,182.044325,176.288166,106.187693,182.357021,118.649261,204.771987,189.863348,134.464073,142.625612,183.606916,139.640504,127.866212
max,79949.000000,0.0,768.000000,768.000000,254.654482,253.695998,253.833737,251.272612,253.824284,253.614965,254.682362,253.999756,201.765755,216.375511,253.885545,239.112386,204.345701


,Unnamed: 0,label,height,width,mean_R,mean_G,mean_B,mean_H,mean_L,mean_S,mean_V,mean_LAB_L,mean_LAB_A,mean_LAB_B,mean_Y,mean_Cr,mean_Cb
count,39975.000000,39975.0,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000,39975.000000
mean,39974.000000,1.0,569.553171,716.566654,161.332810,154.419127,133.904841,69.106661,148.573973,137.351767,180.029285,161.994396,130.470514,139.920640,154.154311,133.122460,116.574152
std,23079.865684,0.0,118.956644,101.020640,43.848655,40.495461,50.819635,33.029889,40.212712,49.526160,38.227437,38.345730,9.775796,15.462648,39.243437,15.323422,17.238611
min,0.000000,1.0,112.000000,320.000000,3.926692,4.089851,0.797363,0.000000,6.866946,0.000000,8.237147,5.844866,52.540156,48.397239,7.201032,29.707807,1.147952
25%,19987.000000,1.0,512.000000,768.000000,133.833668,129.254881,98.536690,43.986773,121.376182,103.243885,157.312074,138.499184,126.087752,130.535018,129.117772,126.411771,107.954678
50%,39974.000000,1.0,512.000000,768.000000,163.415665,156.721370,136.055617,65.010434,148.499405,140.907280,183.798457,164.047429,130.101463,138.307513,155.225373,133.136180,118.481457
75%,59961.000000,1.0,640.000000,768.000000,193.226799,182.819714,171.645973,89.105704,177.010490,174.163402,207.955590,189.084368,134.757108,148.431582,181.681780,141.260131,126.218843
max,79948.000000,1.0,768.000000,768.000000,254.434491,254.886549,252.968859,242.346323,252.968859,254.937467,254.917808,252.986606,215.553397,222.299674,252.968859,238.436712,220.389794


In [ ]:
avg_hist_0 = pd.read_csv("avg_hist_0.csv", index_col=0)
avg_hist_1 = pd.read_csv("avg_hist_1.csv", index_col=0)

display(avg_hist_0)
display(avg_hist_1)
plot_each_channel_by_label_separately(avg_hist_0, avg_hist_1)

In [ ]:
# Long execution, load from csv

folder = 'data/test_ai_augmented'

# Get all image file paths (you can filter by extensions if needed)
file_names = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

# Create the DataFrame
df_test_ai = pd.DataFrame({
    'file_name': file_names,
    'label': 1
})

display(df_test_ai)
print(f"Total files: {len(df_test_ai)}")

_, avg_hist_1_ai = compute_histograms_by_label(df_test_ai)
avg_hist_1_ai.to_csv("avg_hist_1_ai.csv", index=True)

In [ ]:
avg_hist_1_ai = pd.read_csv("avg_hist_1_ai.csv", index_col=0)
display(avg_hist_1_ai)
plot_each_channel_by_label_separately(avg_hist_0, avg_hist_1_ai)